# validation error — Python demo

Numerical companion to the entry [validation error](https://dictionaryofml.org/terms/valerr.html) of the [Dictionary of Applied Machine Learning](https://dictionaryofml.org/): it recomputes what the entry states and prints one line per check.

The entry's weather narrative, carried out on the record itself: the daily air temperature at 06:00 and at 15:00 (UTC) at Krems an der Donau for the days that carry both readings (GeoSphere Austria, station 3805, dataset klima-v2-1h). Each day is one data point: its feature is the 06:00 reading, its label the 15:00 reading of the same day.

Requires NumPy and Matplotlib only, and uses fixed seeds, so the printed numbers reproduce exactly. Generated from [`pythondemos/valerr.py`](https://dictionaryofml.org/terms/valerr.py); CC BY 4.0.

Before running a downloaded copy, scan it: `uvx bandit -q -t B102,B307,B602,B605 valerr.py` (B102,B307,B602,B605 are the exec, eval and shell checks of [bandit](https://bandit.readthedocs.io/)) reports nothing: the script runs no shell command and evaluates no generated code.

In [ ]:
# Notebook shim: the script resolves output paths relative to __file__,
# which a notebook kernel does not define; everything lands in the
# working directory instead.
import os
__file__ = os.path.join(os.getcwd(), "valerr.py")
os.makedirs("pythondemos", exist_ok=True)

In [ ]:
"""
valerr.py -- numerical companion to the entry 'validation error'.

The entry's weather narrative, carried out on the record itself: the daily
air temperature at 06:00 and at 15:00 (UTC) at Krems an der Donau for the
days that carry both readings (GeoSphere Austria, station 3805, dataset
klima-v2-1h).  Each day is one data point: its feature is the 06:00
reading, its label the 15:00 reading of the same day.

The picture the entry opens with shows the days of five years, 2020 to
2024, as a scatterplot of the 06:00 against the 15:00 reading.  Three of them
are the training set -- the same three as in the 'training error' entry --
and the hypothesis is h2, the degree-two polynomial through all three,
the second of the two hypotheses of the 'training set' and 'validation
set' entries.  It interpolates, so its training error is zero.  Three
further days are the validation set, and the average of the polynomial's
squared prediction errors on them is the validation error, 14.73.  It is
far above the zero training error, so it exposes what the training error
hides, and still 45% below the average loss over all 1824 days, 26.93:
three days reveal the overfitting without pinning the risk.

The larger example splits the year: every ninth day, forty days, is the
training set; twenty of the remaining days are the validation set; the
other 305 stay held back and stand in for the risk.  The straight line is
h(x) = 5.97 + 1.04 x, and the three numbers are 13.62, 18.82 and 15.31.
The validation error sits near the risk while the training error is below
it, but for a line the gap is 1.70 and a twenty-day average fluctuates by
about 4.2, so one validation set cannot resolve it.  Raising the degree
makes the gap plain: at degree five the three numbers are 12.41, 16.19 and
15.27, and at degree nine they are 11.55, 18.29 and 20.08.

Finally the fluctuation itself: the validation error is an average of
per-day losses, so it is a random quantity.  Recomputed over 300 draws of
a validation set of each size from the held-back days, its mean stays near
the risk while its spread falls from 8.34 at five days to 1.79 at eighty,
close to the standard deviation of a single day's loss divided by the
square root of the size, corrected for drawing without replacement.

Deterministic: fixed seed for the draws; every fit is the closed-form
least-squares solution and the subsets of the later examples are fixed
strides through 2024.
Self-contained: numpy + matplotlib only (stdlib urllib for the download).

Blocks
------
[B-fetch]   Download the 06:00 and the 15:00 temperature at Krems
            for 2020 to 2024; write the records to valerr_weather.csv and
            check them against the archive.
[B-opening] The entry's opening figure: the days of five years, three of
            them the training set, the interpolating degree-two
            polynomial h2 learned from them (training error zero), and
            three validation days.  The validation error is the average
            of the polynomial's three squared prediction errors and
            lands on its average loss over the five years; write the
            points, the curve and the errors for the figure.
[B-def]     The validation error of the line fitted to the forty training
            days: the average of twenty squared errors, with the line
            fixed.  Prints the line itself and checks it against the
            closed-form least-squares solution, so it is visibly the ERM
            solution.
[B-risk]    The validation error against the risk, for degrees 1, 5 and 9:
            the training error is below the risk at every degree, and the
            gap grows with the degree, while for the line it is smaller
            than the fluctuation of a twenty-day average.  The risk here
            is the average loss over the 306 held-back days, and the
            block prints how tightly those 306 days pin it.
[B-spread]  The validation error recomputed over 300 draws of a validation
            set of each size: the spread shrinks as the set grows.  Writes
            the CSV behind the entry's second figure.
[B-cond]    The learned hypothesis is itself random: over 300 draws of a
            training set of forty days the ERM quadratic changes every
            time.  The validation error on twenty days
            drawn independently of the training set still deviates from
            the risk of that draw's hypothesis by zero on average, while
            a validation set that shares ten days with the training set,
            and the training error itself, sit below it.

Outputs
-------
valerr_weather.csv      : date, t06, t15 of the 1824 days that carry
                          both readings
valerr_open_days.csv    : t06, t15 of the days of 2020 to 2024
valerr_open_days_plot.csv : the 20 of them (every 92nd) the figure draws
valerr_open_line.csv    : t06, t15 along the linear hypothesis h1 of the
                          opening figure
valerr_open_train.csv   : t06, t15 of the three training days
valerr_open_val.csv     : t06, t15 of the five validation days
valerr_open_tree.csv    : t06, t15 along the step function of the tree
valerr_open_err.csv     : t06, tree, errplus, errminus -- the vertical
                          gap between each validation day and the tree
valerr_spread.csv       : size, mean, spread of the validation error over
                          300 draws of a validation set of that size
valerr_risk.csv         : size, risk -- the reference level for the figure
valerr_cond.csv         : estimate, mean, spread -- deviation from the risk
                          of the learned hypothesis over 300 draws
valerr.png              : preview (checking only; also shown on screen
                          when the demo is run from a terminal or a
                          notebook)
"""

# /// script
# requires-python = ">=3.10"
# dependencies = ["matplotlib==3.10.7", "numpy==2.3.4"]
# ///

import json
import sys
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib

# Run headless by default: the Agg backend needs no display and keeps the
# saved preview byte-identical across runs, which the committed PNG relies
# on.  When the demo is run from a terminal or inside a notebook, the
# default backend is kept instead, so the figures are shown as well as
# written.
INTERACTIVE = sys.stdout.isatty() or "ipykernel" in sys.modules
if not INTERACTIVE:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT_DIR = Path(__file__).parent

report = []


def check(name, ok):
    report.append((name, bool(ok)))
    print(f"  [{'ok' if ok else 'FAIL'}] {name}")


def avg_sqerr(x, y, coef):
    return float(np.mean((y - np.polyval(coef, x)) ** 2))

**[B-fetch]** Download the 06:00 and the 15:00 temperature at Krems for 2020 to 2024; write the records to valerr_weather.csv and check them against the archive.

In [ ]:
print("[B-fetch] the record")
URL = ("https://dataset.api.hub.geosphere.at/v1/station/historical/"
       "klima-v2-1h?parameters=tl&station_ids=3805"
       "&start=2020-01-01T00:00&end=2024-12-31T23:00")
with urllib.request.urlopen(URL, timeout=300) as resp:
    payload = json.load(resp)
readings = payload["features"][0]["properties"]["parameters"]["tl"]["data"]
byday = {}
for stamp, value in zip(payload["timestamps"], readings):
    if value is None:                      # the archive drops whole days
        continue
    hour = stamp[11:16]                    # the stamps are UTC, and so are
    if hour in ("06:00", "15:00"):         # the two times, so no conversion
        byday.setdefault(stamp[:10], {})[hour] = float(value)
dates5 = [d for d in sorted(byday) if len(byday[d]) == 2]
t06all = np.array([byday[d]["06:00"] for d in dates5])
t15all = np.array([byday[d]["15:00"] for d in dates5])
with open(OUT_DIR / "valerr_weather.csv", "w") as f:
    f.write("date,t06,t15\n")
    for day, lo, hi in zip(dates5, t06all, t15all):
        f.write(f"{day},{lo},{hi}\n")
print(f"    {len(dates5)} days, 06:00 from {t06all.min():.1f} to "
      f"{t06all.max():.1f}, 15:00 from {t15all.min():.1f} to "
      f"{t15all.max():.1f}")
check("[B-fetch] 1824 of the days of 2020 to 2024 carry both readings (the "
      "hourly record drops whole days)", len(dates5) == 1824)
check("[B-fetch] every kept day carries both readings",
      not np.isnan(t06all).any() and not np.isnan(t15all).any())
in2024 = np.array([d.startswith("2024") for d in dates5])
dates = [d for d in dates5 if d.startswith("2024")]   # the later blocks use 2024
t06, t15 = t06all[in2024], t15all[in2024]
check("[B-fetch] the record matches the archive (2024-01-01: 4.0 at 06:00 "
      "and 8.0 at 15:00, both UTC)",
      len(dates) == 365 and (dates[0], t06[0], t15[0]) == ("2024-01-01", 4.0, 8.0))

**[B-opening]** The entry's opening figure: the days of five years, three of them the training set, the interpolating degree-two polynomial h2 learned from them (training error zero), and three validation days. The validation error is the average of the polynomial's three squared prediction errors and lands on its average loss over the five years; write the points, the curve and the errors for the figure.

In [ ]:
print("\n[B-opening] the entry's opening figure")


def pick_day(q, pct):
    """Among the days whose 06:00 reading lies within half a degree of the
    q-quantile of all morning minima, the day at the pct-quantile of the
    maxima (the same three training days as in trainerr.py)."""
    target = np.quantile(t06all, q)
    win = np.where(np.abs(t06all - target) <= 0.5)[0]
    order = win[np.argsort(t15all[win], kind="stable")]
    return order[int(round(pct * (len(order) - 1)))]


three = np.array([pick_day(q, p) for q, p in ((0.12, 0.5), (0.55, 0.75), (0.90, 0.5))])
threeval = np.array([pick_day(q, 0.5) for q in (0.04, 0.48, 0.96)])
x3, y3 = t06all[three], t15all[three]
xv, yv = t06all[threeval], t15all[threeval]
order3 = np.argsort(x3)
x3, y3 = x3[order3], y3[order3]
orderv = np.argsort(xv)
xv, yv = xv[orderv], yv[orderv]
coef2 = np.polyfit(x3, y3, 2)               # h2, the interpolating parabola
coef1 = np.polyfit(x3, y3, 1)               # h1, kept for the contrast only

open_trainerr = avg_sqerr(x3, y3, coef2)
open_valerr = avg_sqerr(xv, yv, coef2)
open_risk = avg_sqerr(t06all, t15all, coef2)
for label, xs, ys in (("training", x3, y3), ("validation", xv, yv)):
    for x, y in zip(xs, ys):
        print(f"    {label:>10} day: 06:00 {x:5.1f}, 15:00 {y:5.1f}, "
              f"h2 {float(np.polyval(coef2, x)):6.1f}")
print(f"    h2 = {coef2[2]:.2f} + {coef2[1]:.2f} x {coef2[0]:+.4f} x^2; "
      f"training error {open_trainerr:.1e}, validation error "
      f"{open_valerr:.2f}, risk {open_risk:.2f} over the five years")
print(f"    for the contrast, h1 = {coef1[1]:.2f} + {coef1[0]:.2f} x has "
      f"training error {avg_sqerr(x3, y3, coef1):.2f} and validation error "
      f"{avg_sqerr(xv, yv, coef1):.2f}")
check("[B-opening] h2 passes through the three training days: training "
      "error zero", open_trainerr < 1e-16)
check("[B-opening] no validation day is a training day",
      not set(threeval) & set(three))
check("[B-opening] the validation error is the average of the three squared "
      "prediction errors",
      np.isclose(open_valerr, np.mean((yv - np.polyval(coef2, xv)) ** 2)))
check(f"[B-opening] the validation error {open_valerr:.2f} is large although "
      f"the training error is zero, so it exposes what the training error "
      f"hides", open_valerr > 5.0 and open_trainerr < 1e-16)
check(f"[B-opening] three validation days do not pin the risk "
      f"{open_risk:.2f}: the validation error {open_valerr:.2f} sits "
      f"{100 * (1 - open_valerr / open_risk):.0f}% below it",
      open_valerr < open_risk)
check("[B-opening] the zero training error hides that h2 is the worse of "
      "the two hypotheses on the validation days",
      avg_sqerr(xv, yv, coef2) > 3.0 * avg_sqerr(xv, yv, coef1))
np.savetxt(OUT_DIR / "valerr_open_days.csv", np.stack([t06all, t15all], 1),
           delimiter=",", header="t06,t15", comments="", fmt="%.1f")
# the entry's figure draws every 92nd day, twenty of the 1824; the numbers
# of the entry stay on all days
PLOT_STRIDE = 92
plot_idx = np.arange(0, t06all.size, PLOT_STRIDE)
np.savetxt(OUT_DIR / "valerr_open_days_plot.csv",
           np.stack([t06all[plot_idx], t15all[plot_idx]], 1),
           delimiter=",", header="t06,t15", comments="", fmt="%.1f")
check("[B-opening] the figure draws twenty of the days, every 92nd",
      plot_idx.size == 20)
np.savetxt(OUT_DIR / "valerr_open_train.csv", np.stack([x3, y3], 1),
           delimiter=",", header="t06,t15", comments="", fmt="%.1f")
np.savetxt(OUT_DIR / "valerr_open_val.csv", np.stack([xv, yv], 1),
           delimiter=",", header="t06,t15", comments="", fmt="%.1f")
lo, hi = t06all.min() - 1.0, t06all.max() + 1.0
xs_poly = np.linspace(lo, hi, 200)
np.savetxt(OUT_DIR / "valerr_open_poly.csv",
           np.stack([xs_poly, np.polyval(coef2, xs_poly)], 1), delimiter=",",
           header="t06,t15", comments="", fmt="%.3f")
np.savetxt(OUT_DIR / "valerr_open_line.csv",
           np.stack([xs_poly, np.polyval(coef1, xs_poly)], 1), delimiter=",",
           header="t06,t15", comments="", fmt="%.3f")
check("[B-opening] the entry's numbers for h1: training error 2.22, "
      "validation error 2.44",
      round(avg_sqerr(x3, y3, coef1), 2) == 2.22
      and round(avg_sqerr(xv, yv, coef1), 2) == 2.44)
with open(OUT_DIR / "valerr_open_err.csv", "w") as f:
    f.write("t06,poly,errplus,errminus\n")
    for x, y in zip(xv, yv):
        on_poly = float(np.polyval(coef2, x))
        gap = y - on_poly
        f.write(f"{x:.2f},{on_poly:.3f},{max(gap, 0.0):.3f},{max(-gap, 0.0):.3f}\n")

**[B-def]** The validation error of the line fitted to the forty training days: the average of twenty squared errors, with the line fixed. Prints the line itself and checks it against the closed-form least-squares solution, so it is visibly the ERM solution.

In [ ]:
print("\n[B-def] the validation error of a fitted line")
idx = np.arange(len(t06))
train = idx[::9][:40]                       # every ninth day, forty of them
rest = np.setdiff1d(idx, train)
val = rest[::16][:20]                       # twenty of the remaining days
held = np.setdiff1d(rest, val)              # the other 306, standing in for the risk
line = np.polyfit(t06[train], t15[train], 1)
losses = (t15[val] - np.polyval(line, t06[val])) ** 2
valerr = avg_sqerr(t06[val], t15[val], line)
print(f"    {len(train)} training days, {len(val)} validation days, "
      f"{len(held)} held back")
print(f"    the line is h(x) = {line[1]:.2f} + {line[0]:.2f} x, the "
      f"least-squares fit to the {len(train)} training days")
# the same line computed in closed form, to show that polyfit is ERM with
# the squared error loss over the linear model and nothing else
design = np.column_stack([t06[train], np.ones(len(train))])
closed = np.linalg.solve(design.T @ design, design.T @ t15[train])
check("[B-def] the line is the ERM solution: polyfit matches the closed-form "
      "least-squares solution", np.allclose(line, closed))
print(f"    validation error {valerr:.2f} = average of {len(val)} squared "
      f"errors (smallest {losses.min():.2f}, largest {losses.max():.2f})")
check("[B-def] the validation error is the average of the per-day losses",
      np.isclose(valerr, float(np.mean(losses)), atol=1e-12))
check("[B-def] the line is fixed: computing the validation error changed no "
      "coefficient", np.allclose(line, np.polyfit(t06[train], t15[train], 1)))
check("[B-def] no validation day is a training day", not set(val) & set(train))

**[B-risk]** The validation error against the risk, for degrees 1, 5 and 9: the training error is below the risk at every degree, and the gap grows with the degree, while for the line it is smaller than the fluctuation of a twenty-day average. The risk here is the average loss over the 306 held-back days, and the block prints how tightly those 306 days pin it.

In [ ]:
print("\n[B-risk] the two errors against the risk")
three = []
for degree in (1, 5, 9):
    coef = np.polyfit(t06[train], t15[train], degree)
    three.append((degree, avg_sqerr(t06[train], t15[train], coef),
                  avg_sqerr(t06[val], t15[val], coef),
                  avg_sqerr(t06[held], t15[held], coef)))
    print(f"    degree {degree}: training error {three[-1][1]:6.2f}, validation "
          f"error {three[-1][2]:6.2f}, risk {three[-1][3]:8.2f}")
per_day = (t15[held] - np.polyval(line, t06[held])) ** 2
noise = float(per_day.std()) / np.sqrt(len(val))
# the risk itself is an expectation and is not computable from a finite
# record; every "risk" printed here is the average loss over the held-back
# days, which is an average of that many numbers and carries its own spread
held_noise = float(per_day.std()) / np.sqrt(len(held))
print(f"    the risk column is the average loss over the {len(held)} "
      f"held-back days, {three[0][3]:.2f} for the line, itself uncertain "
      f"by about {held_noise:.2f}")
print(f"    for the line the training error is below the risk by "
      f"{three[0][3] - three[0][1]:.2f}, while a {len(val)}-day average "
      f"fluctuates by about {noise:.2f}")
check("[B-risk] the held-back average pins the risk more tightly than a "
      "validation set of twenty days, by the square root of the ratio of "
      "the two sizes",
      abs(held_noise - noise * np.sqrt(len(val) / len(held)))
      < 0.01 * held_noise)
check("[B-risk] the training error is below the risk at every degree",
      all(t < r for _, t, _, r in three))
check("[B-risk] the gap between the training error and the risk grows with "
      "the degree",
      all(a[3] - a[1] < b[3] - b[1] for a, b in zip(three, three[1:])))
check("[B-risk] for the line that gap is smaller than the fluctuation of a "
      "twenty-day average", three[0][3] - three[0][1] < noise)

**[B-spread]** The validation error recomputed over 300 draws of a validation set of each size: the spread shrinks as the set grows. Writes the CSV behind the entry's second figure.

In [ ]:
print("\n[B-spread] the fluctuation of the validation error")
SIZES = [5, 10, 20, 40, 80]
DRAWS = 300
risk = float(per_day.mean())
gen = np.random.default_rng(0)
rows = []
for size in SIZES:
    means = [float(per_day[gen.choice(len(per_day), size, replace=False)].mean())
             for _ in range(DRAWS)]
    rows.append((size, float(np.mean(means)), float(np.std(means))))
    # the days are drawn without replacement from a finite pool, so the
    # spread of the average carries the finite-population correction
    expected = (per_day.std() / np.sqrt(size)
                * np.sqrt((len(per_day) - size) / (len(per_day) - 1)))
    print(f"    {size:>2} validation days: mean {rows[-1][1]:6.2f}, spread "
          f"{rows[-1][2]:5.2f} (a single day's loss has spread "
          f"{per_day.std():.2f}, so {expected:5.2f} is expected)")
check("[B-spread] the spread shrinks as the validation set grows",
      all(a[2] > b[2] for a, b in zip(rows, rows[1:])))
check("[B-spread] each mean stays within 10% of the risk",
      all(abs(m - risk) < 0.10 * risk for _, m, _ in rows))
def expected_spread(n):
    """The spread of an average of n of the held-back losses, drawn without
    replacement: one day's spread over the square root of n, times the
    finite-population correction."""
    return (per_day.std() / np.sqrt(n)
            * np.sqrt((len(per_day) - n) / (len(per_day) - 1)))


check("[B-spread] each spread is within 20% of the standard deviation of a "
      "single day's loss divided by the square root of the size, corrected "
      "for drawing without replacement (the 300 draws leave this much "
      "sampling noise of their own)",
      all(abs(s - expected_spread(n)) < 0.20 * expected_spread(n)
          for n, _, s in rows))

with open(OUT_DIR / "valerr_spread.csv", "w") as fh:
    fh.write("size,mean,spread\n")
    for n, m, s in rows:
        fh.write(f"{n},{m:.4f},{s:.4f}\n")
with open(OUT_DIR / "valerr_risk.csv", "w") as fh:
    fh.write("size,risk\n")
    fh.write(f"{SIZES[0]},{risk:.4f}\n")
    fh.write(f"{SIZES[-1]},{risk:.4f}\n")

**[B-cond]** The learned hypothesis is itself random: over 300 draws of a training set of forty days the ERM quadratic changes every time. The validation error on twenty days drawn independently of the training set still deviates from the risk of that draw's hypothesis by zero on average, while a validation set that shares ten days with the training set, and the training error itself, sit below it.

In [ ]:
print("\n[B-cond] the validation error of a hypothesis that is itself random")
# the 366 days stand in for the probability distribution of a day; a draw
# consists of a random training set of forty days, the ERM quadratic
# learned from it, and twenty validation days drawn from the
# rest of the year (so independent of the training set) -- beside a leaky
# validation set in which ten of the twenty days are training days
DEGREE_C, DRAWS_C = 2, 300
gen_c = np.random.default_rng(3)
dev = {"independent validation set": [], "validation set sharing ten "
       "training days": [], "training set": []}
for _ in range(DRAWS_C):
    perm = gen_c.permutation(len(t06))
    tr, fresh = perm[:40], perm[40:]
    coef = np.polyfit(t06[tr], t15[tr], DEGREE_C)   # changes with every draw
    risk_h = avg_sqerr(t06, t15, coef)               # risk of this draw's hypothesis
    val_ind = fresh[:20]
    val_leaky = np.concatenate([tr[:10], fresh[20:30]])
    dev["independent validation set"].append(
        avg_sqerr(t06[val_ind], t15[val_ind], coef) - risk_h)
    dev["validation set sharing ten training days"].append(
        avg_sqerr(t06[val_leaky], t15[val_leaky], coef) - risk_h)
    dev["training set"].append(avg_sqerr(t06[tr], t15[tr], coef) - risk_h)
crows = []
for name, d in dev.items():
    d = np.array(d)
    crows.append((name, float(d.mean()), float(d.std()),
                  float(d.std() / np.sqrt(DRAWS_C))))
    print(f"    {name:>42}: mean deviation {d.mean():+6.2f}, spread "
          f"{d.std():5.2f}, standard error of the mean {crows[-1][3]:.2f}")
check("[B-cond] on the independent validation set the deviation from the "
      "risk of the learned hypothesis averages to zero: within three "
      "standard errors", abs(crows[0][1]) < 3.0 * crows[0][3])
check("[B-cond] the leaky validation set and the training set both sit below "
      "the risk by more than three standard errors",
      crows[1][1] < -3.0 * crows[1][3] and crows[2][1] < -3.0 * crows[2][3])
check("[B-cond] the training set is the more optimistic of the two",
      crows[2][1] < crows[1][1])
with open(OUT_DIR / "valerr_cond.csv", "w") as fh:
    fh.write("estimate,mean,spread\n")
    for name, m, sd, _ in crows:
        fh.write(f"{name},{m:.4f},{sd:.4f}\n")


# ---- preview (checking only)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.0))
ax = axes[0]
ax.plot(t06all, t15all, ".", color="0.7", ms=3, label="days of 2020 to 2024")
ax.plot(xs_poly, np.polyval(coef2, xs_poly), "k--", lw=1.4,
        label="h2, the interpolating parabola")
for x, y in zip(xv, yv):
    ax.plot([x, x], [float(np.polyval(coef2, x)), y], "--", color="red", lw=1.0)
ax.plot(x3, y3, "o", color="black", ms=8, label="training set (3 days)")
ax.plot(xv, yv, "^", mfc="none", mec="red", ms=9, mew=1.5,
        label="validation set (3 days)")
ax.set_ylim(t15all.min() - 12.0, t15all.max() + 4.0)
ax.set_xlabel("temperature at 06:00 UTC (feature, °C)")
ax.set_ylabel("temperature at 15:00 UTC (label, °C)")
ax.set_title(f"Five years at Krems: training error {open_trainerr:.0f}, "
             f"validation error {open_valerr:.2f}, risk {open_risk:.2f}",
             fontsize=9)
ax.legend(frameon=False, fontsize=8, loc="upper left")
ax = axes[1]
ax.errorbar([r[0] for r in rows], [r[1] for r in rows],
            yerr=[r[2] for r in rows], fmt="o", ms=5, color="black",
            capsize=3, label="validation error (mean and spread)")
ax.axhline(risk, ls="--", color="0.4", label="risk of the fixed line")
ax.set_xscale("log")
ax.set_xticks(SIZES)
ax.set_xticklabels([str(s) for s in SIZES])
ax.set_xlabel("number of validation days")
ax.set_ylabel("validation error")
ax.set_title("The average fluctuates less as the validation set grows",
             fontsize=9)
ax.legend(frameon=False, fontsize=8)
ax = axes[2]
ax.errorbar(range(3), [r[1] for r in crows], yerr=[r[2] for r in crows],
            fmt="o", ms=5, color="black", capsize=3,
            label="mean and spread over 300 draws")
ax.axhline(0.0, ls="--", color="0.4", label="risk of the learned hypothesis")
ax.set_xticks(range(3))
ax.set_xticklabels(["independent\nvalidation set", "validation set\nsharing "
                    "10 training days", "training set"], fontsize=8)
ax.set_xlabel("set the average loss is taken over")
ax.set_ylabel("average loss minus risk of the learned hypothesis")
ax.set_title("Quadratic learned anew per draw: only the independent "
             "validation set is unbiased", fontsize=9)
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(OUT_DIR / "valerr.png", dpi=150)
if INTERACTIVE:
    plt.show()
else:
    plt.close(fig)

passed = sum(1 for _, ok in report if ok)
print(f"\n{passed}/{len(report)} checks pass")